# 09 - Feature Engineering v3 (Opsi B)

Menambahkan fitur rasio custom (debt/credit, application/credit, payment ratio,
days past due, balance/limit) dan fitur recent-window 6 bulan di atas fondasi
v2, karena Feature Engineering v2 (Opsi A) sudah mentok di ~0.78 dan hyperparameter
tuning terbukti tidak banyak membantu.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from modeling.train import load_featured_dataset, load_feature_metadata
from cleaning_auxiliary import load_all_auxiliary_tables
from features_v2 import merge_with_main_dataset
from features_v3 import build_features_v3, save_featured_dataset_v3, build_feature_metadata_v3, save_feature_metadata_v3
import json


## Load Dataset Utama v1 (BUKAN v2) dan 6 Tabel Tambahan

v3 dibangun ulang dari fondasi v1 + tabel mentah -- bukan ditempel di atas v2 --
supaya tidak ada duplikasi/bentrok nama kolom.

In [2]:
df_main = load_featured_dataset()  # v1, 73 kolom
auxiliary_tables = load_all_auxiliary_tables()

with open("../data/processed/feature_engineering_metadata_v2.json") as f:
    metadata_v2 = json.load(f)

print("Shape dataset utama (v1):", df_main.shape)
print("Jumlah fitur v2 sebelumnya:", len(metadata_v2["tree_features_v2"]))

Shape dataset utama (v1): (307511, 73)
Jumlah fitur v2 sebelumnya: 574


## Bangun Fitur v3 (Rasio + Recent Window) dan Gabungkan

In [3]:
features_v3 = build_features_v3(auxiliary_tables)
df_v3 = merge_with_main_dataset(df_main, features_v3)

print("Shape fitur v3 (sebelum digabung):", features_v3.shape)
print("Shape dataset v3 (setelah digabung):", df_v3.shape)

Shape fitur v3 (sebelum digabung): (349684, 674)
Shape dataset v3 (setelah digabung): (307511, 746)


## Simpan Dataset dan Metadata v3

In [4]:
metadata_v3 = build_feature_metadata_v3(df_v3, metadata_v2)

dataset_path = save_featured_dataset_v3(df_v3)
metadata_path = save_feature_metadata_v3(metadata_v3)

print("Jumlah fitur BARU di v3 (di luar v2):", metadata_v3["n_new_features"])
print("Total fitur v3:", len(metadata_v3["tree_features_v3"]))
print("Dataset tersimpan di :", dataset_path)
print("Metadata tersimpan di:", metadata_path)

Jumlah fitur BARU di v3 (di luar v2): 165
Total fitur v3: 739
Dataset tersimpan di : D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\application_train_featured_v3.csv
Metadata tersimpan di: D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\feature_engineering_metadata_v3.json
